# InstructGrid Walkthrough

This notebook shows the full pipeline from the repository in one place: data generation, dataset creation, model construction, training, and evaluation.

It uses a small demo dataset so the steps are easy to follow in Jupyter. If you want the full experiment, use the repository scripts such as `generate_demo_data.py`, `train.py`, and `evaluate.py`.

In [2]:
from pathlib import Path
import json
import random

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import yaml

from grid_env_simple import GridEnv
from generate_demo_data import TYPE_GOALS, INSTRUCTION_CATALOGUE, run_oracle_episode
from utils.tokenizer import Tokenizer
from utils.dataset_utils import DemoDataset
from models import build_model, ALL_MODEL_NAMES
from models.components import derive_colour_labels

MODEL_NAME = "film"  # change to attention, lstm_attention, or film
DEMO_DATA_PATH = Path("notebook_walkthrough_oracle.jsonl")
VOCAB_PATH = Path("notebook_walkthrough_vocab.json")
CONFIG_PATH = Path("configs") / f"{MODEL_NAME}.yaml"

with CONFIG_PATH.open() as f:
    cfg = yaml.safe_load(f)

cfg["epochs"] = 2
cfg["batch_size"] = 16
cfg["augment"] = False
cfg["val_frac"] = 0.2
cfg["seed"] = 42

random.seed(cfg["seed"])
torch.manual_seed(cfg["seed"])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Available models:", ALL_MODEL_NAMES)
print("Selected model:", MODEL_NAME)
print("Config file:", CONFIG_PATH)

Device: cuda
Available models: ['cnn_gru', 'attention', 'lstm_attention', 'film']
Selected model: film
Config file: configs\film.yaml


## 1) Generate a tiny demo dataset

This cell reuses the oracle policy from `generate_demo_data.py` to create a compact JSONL file. Each trajectory starts with an instruction and then stores the state-action transitions that follow it.

In [3]:
def generate_demo_file(out_path: Path, per_type: int = 8, noise: float = 0.10, seed: int = 42):
    env = GridEnv()
    rng = random.Random(seed)
    rows = []

    for type_key in TYPE_GOALS:
        phrasings = INSTRUCTION_CATALOGUE[type_key]
        count = 0
        while count < per_type:
            instruction = rng.choice(phrasings)
            episode_seed = rng.randint(0, 999_999)
            transitions = run_oracle_episode(
                env,
                type_key,
                instruction,
                seed=episode_seed,
                noise=noise,
            )
            if transitions:
                rows.extend(transitions)
                count += 1

    with out_path.open("w") as f:
        for row in rows:
            f.write(json.dumps(row) + "\n")

    return rows

rows = generate_demo_file(DEMO_DATA_PATH, per_type=8, noise=0.10, seed=cfg["seed"])
print(f"Wrote {len(rows)} transitions to {DEMO_DATA_PATH}")
print("First example:")
print(json.dumps(rows[0], indent=2))

Wrote 477 transitions to notebook_walkthrough_oracle.jsonl
First example:
{
  "instruction": "put the red box on the red target",
  "state": {
    "agent_pos": [
      1,
      4
    ],
    "red_box_pos": [
      5,
      1
    ],
    "blue_box_pos": [
      4,
      4
    ],
    "red_tgt_pos": [
      7,
      7
    ],
    "blue_tgt_pos": [
      4,
      1
    ],
    "held_object": null
  },
  "action": 1,
  "next_state": {
    "agent_pos": [
      2,
      4
    ],
    "red_box_pos": [
      5,
      1
    ],
    "blue_box_pos": [
      4,
      4
    ],
    "red_tgt_pos": [
      7,
      7
    ],
    "blue_tgt_pos": [
      4,
      1
    ],
    "held_object": null
  }
}


## 2) Build the tokenizer and dataset

The tokenizer is word-level and is built from the instruction catalogue so it covers every phrasing in the repository. `DemoDataset` then turns each JSONL transition into the three tensors used by the models: token ids, grid tensor, and action label.

In [4]:
tok = Tokenizer()
tok.build_from_catalogue(INSTRUCTION_CATALOGUE)
tok.save(str(VOCAB_PATH))
print("Tokenizer vocab size:", tok.vocab_size)

train_ds, val_ds = DemoDataset.trajectory_split(
    [str(DEMO_DATA_PATH)],
    tok,
    max_len=cfg["max_len"],
    val_frac=cfg["val_frac"],
    augment=cfg["augment"],
    seed=cfg["seed"],
)

train_loader = DataLoader(train_ds, batch_size=cfg["batch_size"], shuffle=True, num_workers=0)
val_loader = DataLoader(val_ds, batch_size=cfg["batch_size"], shuffle=False, num_workers=0)

batch_tokens, batch_grid, batch_actions = next(iter(train_loader))
print("token batch shape:", batch_tokens.shape)
print("grid batch shape:", batch_grid.shape)
print("actions shape:", batch_actions.shape)
print("decoded example:", tok.decode(batch_tokens[0]))
print("first action label:", int(batch_actions[0]))

[Tokenizer] vocab size (catalogue): 20
Tokenizer vocab size: 20
[Dataset] 477 transitions  | 32 trajectories  | 1 file(s)
[Dataset] train: 382 steps (26 trajs)  |  val: 95 steps (6 trajs)
token batch shape: torch.Size([16, 20])
grid batch shape: torch.Size([16, 7, 8, 8])
actions shape: torch.Size([16])
decoded example: place the blue box in the blue zone
first action label: 1


In [17]:
import numpy as np
np.unique(batch_actions)

array([0, 1, 2, 3, 4, 5], dtype=int64)

In [18]:
batch_actions

tensor([1, 0, 2, 3, 3, 1, 1, 5, 1, 0, 4, 0, 2, 0, 1, 4])

In [14]:
batch_grid[0][3]

tensor([[0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0., 0., 0., 0.]])

In [10]:
batch_tokens[15]

tensor([15,  3, 19,  5, 13,  3,  4,  7,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
         0,  0])

## 3) Create the model and inspect one forward pass

`build_model()` is the same registry-based factory used by the training script. The model takes `(tokens, grid)` and returns a dictionary with action logits and, for some models, auxiliary colour logits.

In [ ]:
model = build_model(MODEL_NAME, tok.vocab_size, cfg).to(device)
print(model.__class__.__name__)
print("Number of parameters:", sum(p.numel() for p in model.parameters()))

batch_tokens = batch_tokens.to(device)
batch_grid = batch_grid.to(device)

with torch.no_grad():
    out = model(batch_tokens, batch_grid)

print("Model outputs:", out.keys())
print("action logits shape:", out["action"].shape)
if out.get("colour") is not None:
    print("colour logits shape:", out["colour"].shape)

decoded = [tok.decode(batch_tokens[i].cpu()) for i in range(min(4, batch_tokens.size(0)))]
print("Colour labels derived from instructions:", derive_colour_labels(decoded, tok).tolist())

FiLMAgent
Number of parameters: 917832
Model outputs: dict_keys(['action', 'colour'])
action logits shape: torch.Size([16, 6])
colour logits shape: torch.Size([16, 2])
Colour labels derived from instructions: [1, 1, 0, 0]


In [25]:
decoded

['place the blue box in the blue zone',
 'take the blue box to the red zone',
 'put the red block on the blue target',
 'take the red box to the red zone']

In [22]:
out.keys()

dict_keys(['action', 'colour'])

In [ ]:
out['action'][0]

tensor([-1.1969,  0.1443,  0.6078,  0.2531,  0.1841,  0.3848], device='cuda:0')

In [24]:
out['colour']

tensor([[ 0.0274,  0.1266],
        [-0.0024,  0.1373],
        [ 0.0149,  0.1330],
        [ 0.0149,  0.1076],
        [ 0.0078,  0.1490],
        [ 0.0257,  0.1277],
        [ 0.0174,  0.1476],
        [ 0.0186,  0.1011],
        [ 0.0154,  0.1419],
        [ 0.0162,  0.1547],
        [ 0.0239,  0.1789],
        [ 0.0190,  0.1076],
        [ 0.0043,  0.1399],
        [ 0.0347,  0.1414],
        [ 0.0207,  0.1010],
        [-0.0026,  0.1073]], device='cuda:0')

## 4) Training setup

This cell mirrors the training script: weighted cross-entropy for the action head, optional auxiliary colour loss, AdamW, and cosine learning-rate decay.

In [26]:
def build_criterion(cfg, device):
    move_weight = cfg.get("move_weight", 1.0)
    pick_weight = cfg.get("pick_weight", 6.0)
    place_weight = cfg.get("place_weight", 8.0)
    label_smoothing = cfg.get("label_smoothing", 0.1)
    weights = torch.tensor([move_weight, move_weight, move_weight, move_weight, pick_weight, place_weight], dtype=torch.float32).to(device)
    return nn.CrossEntropyLoss(weight=weights, label_smoothing=label_smoothing)

criterion = build_criterion(cfg, device)
aux_ce = nn.CrossEntropyLoss(ignore_index=-1)
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg["lr"], weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["epochs"])
lambda_colour = cfg.get("lambda_colour", 0.3)
is_lstm = MODEL_NAME == "lstm_attention"

def run_epoch(model, loader, train_mode: bool):
    model.train() if train_mode else model.eval()
    total_loss = 0.0
    total_correct = 0.0
    total_items = 0
    ctx = torch.enable_grad() if train_mode else torch.no_grad()

    with ctx:
        for tokens, grid, actions in loader:
            tokens = tokens.to(device)
            grid = grid.to(device)
            actions = actions.to(device)

            if is_lstm:
                out = model(tokens, grid, hx=None)
            else:
                out = model(tokens, grid)

            action_logits = out["action"]
            colour_logits = out.get("colour")
            loss = criterion(action_logits, actions)

            if colour_logits is not None and lambda_colour > 0:
                instructions = [tok.decode(tokens[i].cpu()) for i in range(tokens.size(0))]
                clr_labels = derive_colour_labels(instructions, tok).to(device)
                aux_loss = aux_ce(colour_logits, clr_labels)
                if not torch.isnan(aux_loss):
                    loss = loss + lambda_colour * aux_loss

            if train_mode:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            batch_size = actions.size(0)
            total_loss += loss.item() * batch_size
            total_correct += (action_logits.argmax(1) == actions).float().sum().item()
            total_items += batch_size

    return total_loss / total_items, total_correct / total_items

print("Criterion, optimizer, and scheduler are ready.")

Criterion, optimizer, and scheduler are ready.


## 5) Train for a couple of epochs

This is the same loop pattern as `train.py`, just shortened so it is easy to run inside a notebook.

In [ ]:
best_val_loss = float("inf")

for epoch in range(1, cfg["epochs"] + 1):
    train_loss, train_acc = run_epoch(model, train_loader, True)
    val_loss, val_acc = run_epoch(model, val_loader, False)
    scheduler.step()

    print(
        f"Epoch {epoch:2d}/{cfg['epochs']}  "
        f"train loss={train_loss:.4f} acc={train_acc:.3f}  "
        f"val loss={val_loss:.4f} acc={val_acc:.3f}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss

print("Best validation loss:", round(best_val_loss, 4))

Epoch  1/2  train loss=1.9991 acc=0.115  val loss=2.0696 acc=0.074
Best validation loss: 2.0696


## 6) Evaluate the trained model

This headless evaluation follows the same idea as `evaluate.py`: reset the grid, let the model act step by step, and stop when the instruction goal is satisfied.

In [28]:
def goal_met(instruction: str, state: dict) -> bool:
    instr = instruction.lower()
    rp = list(state["red_box_pos"])
    bp = list(state["blue_box_pos"])
    rt = list(state["red_tgt_pos"])
    bt = list(state["blue_tgt_pos"])
    held = state.get("held_object")

    red_box = "red box" in instr or "red block" in instr
    blue_box = "blue box" in instr or "blue block" in instr
    red_zone = "red zone" in instr or "red target" in instr or "red area" in instr
    blue_zone = "blue zone" in instr or "blue target" in instr or "blue area" in instr

    if red_box and red_zone and not blue_zone:
        return held != "red" and rp == rt
    if red_box and blue_zone and not red_zone:
        return held != "red" and rp == bt
    if blue_box and blue_zone and not red_zone:
        return held != "blue" and bp == bt
    if blue_box and red_zone and not blue_zone:
        return held != "blue" and bp == rt
    return rp == rt and bp == bt

def run_episode(instruction: str, seed: int = 0, max_steps: int = 60):
    env = GridEnv()
    state = env.reset(seed=seed)
    tokens = tok.encode(instruction).unsqueeze(0).to(device)

    for step in range(max_steps):
        grid = env.get_tensor(state).unsqueeze(0).to(device)
        with torch.no_grad():
            if is_lstm:
                action, _ = model.predict(tokens, grid, hx=None)
            else:
                action, _ = model.predict(tokens, grid)

        state, _, done, info = env.step(action)
        if goal_met(instruction, state):
            return True, step + 1, info
        if done:
            break

    return goal_met(instruction, state), info.get("steps", max_steps), info

demo_instructions = [
    "place the red box in the red zone",
    "place the blue box in the blue zone",
    "place the red box in the blue zone",
    "place the blue box in the red zone",
]

for i, instruction in enumerate(demo_instructions):
    success, steps, info = run_episode(instruction, seed=100 + i)
    status = "SUCCESS" if success else "FAIL"
    print(f"{status:7s} | {instruction:<35s} | steps={steps:2d} | held={info.get('held')}")

FAIL    | place the red box in the red zone   | steps=60 | held=None
FAIL    | place the blue box in the blue zone | steps=60 | held=None
FAIL    | place the red box in the blue zone  | steps=60 | held=None
FAIL    | place the blue box in the red zone  | steps=60 | held=None


## What this notebook covered

- How the repository generates instruction-following trajectories.
- How `Tokenizer` converts text into token ids.
- How `DemoDataset` converts JSONL transitions into training tensors.
- How `build_model()` creates one of the repository models.
- How the training loop combines action loss and auxiliary colour loss.
- How evaluation rolls the model through the environment step by step.